# 03 · Mô hình hợp nhất đa phương thức theo thời gian (C1)

**Mục tiêu:** từ vector 54 đặc trưng mỗi 2 giây (khuôn mặt, danh tính, vật thể, âm thanh, trình duyệt, nhịp gõ, bài làm,
hệ thống) dự đoán xác suất gian lận **theo dòng thời gian (streaming)**, học được từ **nhãn yếu** (chỉ biết đoạn/phiên có
gian lận), và **chịu được khi thiếu phương thức** (máy yếu, không có micro, chưa có YOLO…).

**Kiến trúc** (`eg_common.build_fusion_model`): encoder riêng từng phương thức → cổng hợp nhất (che phương thức thiếu)
→ GRU nhân quả (mỗi bước = một cửa sổ) → xác suất. Xuất ONNX **một bước** (`x, h → prob, h_out, gates`) để server chạy
streaming; cổng `gates` dùng làm giải thích cho giám thị.

**Dữ liệu:**
1. **OEP** (Atoum et al., 24 người, video webcam + nhãn khoảng thời gian gian lận) — trích đặc trưng khuôn mặt (BlazeFace),
   âm thanh (Silero VAD), vật thể (YOLO từ notebook 02) bằng đúng công thức của trình duyệt. Tải từ Kaggle (cần `kaggle.json`).
   Trang gốc: cvlab.cse.msu.edu/oep-dataset.html — hãy trích dẫn bài báo OEP khi dùng.
2. **Phiên thi tổng hợp** có chèn đoạn gian lận (rời tab tra cứu, điện thoại, người nhắc, màn hình phụ, dán, chép câu trả lời LLM)
   và các tình huống vô hại dễ báo nhầm (thông báo hệ thống, ho, mất mạng, máy yếu). Cửa sổ nhịp gõ lấy từ dữ liệu thật
   của notebook 01 nếu có. **Nêu rõ trong bài báo phần nào dùng dữ liệu tổng hợp.**
3. (Tùy chọn) Dữ liệu xuất từ trang giám thị ExamGuard (đặc trưng + nhãn xác nhận/bác cờ, mã giả danh) để tinh chỉnh.

**Đánh giá:** chia theo người; AUC, recall @ FPR 1% / 5% mức cửa sổ và mức phiên; so với baseline luật ngưỡng (đang chạy
trên server khi chưa có mô hình); ablation từng phương thức, bỏ modality dropout, bỏ cổng, bỏ tiền huấn luyện;
ngưỡng theo hạng máy (C5).

Thời gian: không có OEP ~15 phút (GPU T4); có OEP thêm ~30–60 phút trích đặc trưng.

In [ ]:
!pip -q install mediapipe silero-vad onnx onnxruntime kaggle opencv-python-headless pyarrow

### Mã dùng chung
Hai ô dưới ghi `feature_schema.json` (hợp đồng đặc trưng với trang web) và `eg_common.py` (công thức đặc trưng, baseline luật, mô hình hợp nhất). Không cần sửa.

In [ ]:
%%writefile feature_schema.json
{
 "version": 1,
 "window_seconds": 2,
 "modalities": [
  "face",
  "identity",
  "object",
  "audio",
  "browser",
  "keystroke",
  "answer",
  "system"
 ],
 "features": [
  {
   "name": "face_count_mean",
   "modality": "face",
   "scale": 1,
   "source": "client",
   "desc": "Số khuôn mặt trung bình"
  },
  {
   "name": "face_count_max",
   "modality": "face",
   "scale": 1,
   "source": "client",
   "desc": "Số khuôn mặt lớn nhất"
  },
  {
   "name": "face_missing_ratio",
   "modality": "face",
   "scale": 1,
   "source": "client",
   "desc": "Tỉ lệ mẫu không thấy mặt"
  },
  {
   "name": "yaw_mean",
   "modality": "face",
   "scale": 45,
   "source": "client",
   "desc": "Góc quay ngang trung bình (độ, ước lượng từ 6 điểm)"
  },
  {
   "name": "yaw_std",
   "modality": "face",
   "scale": 20,
   "source": "client",
   "desc": "Độ lệch chuẩn góc quay ngang"
  },
  {
   "name": "pitch_mean",
   "modality": "face",
   "scale": 30,
   "source": "client",
   "desc": "Góc ngẩng/cúi trung bình (độ)"
  },
  {
   "name": "pitch_std",
   "modality": "face",
   "scale": 15,
   "source": "client",
   "desc": "Độ lệch chuẩn góc ngẩng/cúi"
  },
  {
   "name": "head_away_ratio",
   "modality": "face",
   "scale": 1,
   "source": "client",
   "desc": "Tỉ lệ mẫu quay đầu quá ngưỡng (|yaw|>30° hoặc |pitch|>25°)"
  },
  {
   "name": "box_shift",
   "modality": "face",
   "scale": 0.2,
   "source": "client",
   "desc": "Dịch chuyển tâm hộp mặt (theo chiều rộng khung)"
  },
  {
   "name": "face_size",
   "modality": "face",
   "scale": 0.3,
   "source": "client",
   "desc": "Diện tích hộp mặt / diện tích khung"
  },
  {
   "name": "face_conf",
   "modality": "face",
   "scale": 1,
   "source": "client",
   "desc": "Độ tin cậy dò mặt trung bình"
  },
  {
   "name": "id_score",
   "modality": "identity",
   "scale": 1,
   "source": "server",
   "desc": "Điểm khớp gần nhất với ảnh đăng ký"
  },
  {
   "name": "id_checked",
   "modality": "identity",
   "scale": 1,
   "source": "server",
   "desc": "Có lần xác thực lại trong cửa sổ"
  },
  {
   "name": "id_mismatch",
   "modality": "identity",
   "scale": 1,
   "source": "server",
   "desc": "Lần xác thực gần nhất không khớp"
  },
  {
   "name": "obj_phone",
   "modality": "object",
   "scale": 1,
   "source": "server",
   "desc": "Xác suất điện thoại"
  },
  {
   "name": "obj_book",
   "modality": "object",
   "scale": 1,
   "source": "server",
   "desc": "Xác suất sách/giấy"
  },
  {
   "name": "obj_headphone",
   "modality": "object",
   "scale": 1,
   "source": "server",
   "desc": "Xác suất tai nghe"
  },
  {
   "name": "obj_person_extra",
   "modality": "object",
   "scale": 1,
   "source": "server",
   "desc": "Xác suất có người thứ hai (theo thân người)"
  },
  {
   "name": "obj_screen",
   "modality": "object",
   "scale": 1,
   "source": "server",
   "desc": "Xác suất màn hình/laptop phụ"
  },
  {
   "name": "obj_age",
   "modality": "object",
   "scale": 30,
   "source": "server",
   "desc": "Số giây kể từ khung gần nhất (độ tươi)"
  },
  {
   "name": "speech_ratio",
   "modality": "audio",
   "scale": 1,
   "source": "client",
   "desc": "Tỉ lệ thời gian có tiếng nói"
  },
  {
   "name": "speech_segments",
   "modality": "audio",
   "scale": 3,
   "source": "client",
   "desc": "Số đoạn nói bắt đầu trong cửa sổ"
  },
  {
   "name": "audio_energy",
   "modality": "audio",
   "scale": 1,
   "source": "client",
   "desc": "Năng lượng âm thanh (0..1, thang dB chuẩn hóa)"
  },
  {
   "name": "hidden_sec",
   "modality": "browser",
   "scale": 2,
   "source": "client",
   "desc": "Số giây trang bị ẩn (rời tab)"
  },
  {
   "name": "blur_sec",
   "modality": "browser",
   "scale": 2,
   "source": "client",
   "desc": "Số giây cửa sổ mất focus"
  },
  {
   "name": "tab_switches",
   "modality": "browser",
   "scale": 1,
   "source": "client",
   "desc": "Số lần rời tab/cửa sổ"
  },
  {
   "name": "fullscreen_exits",
   "modality": "browser",
   "scale": 1,
   "source": "client",
   "desc": "Số lần thoát toàn màn hình"
  },
  {
   "name": "not_fullscreen_sec",
   "modality": "browser",
   "scale": 2,
   "source": "client",
   "desc": "Số giây không ở chế độ toàn màn hình"
  },
  {
   "name": "window_small",
   "modality": "browser",
   "scale": 1,
   "source": "client",
   "desc": "Cửa sổ bị thu nhỏ/chia đôi (0/1)"
  },
  {
   "name": "screen_extended",
   "modality": "browser",
   "scale": 1,
   "source": "client",
   "desc": "Đang có màn hình mở rộng (0/1, chỉ Chromium)"
  },
  {
   "name": "screen_changes",
   "modality": "browser",
   "scale": 1,
   "source": "client",
   "desc": "Số lần thay đổi cấu hình màn hình"
  },
  {
   "name": "paste_attempts",
   "modality": "browser",
   "scale": 1,
   "source": "client",
   "desc": "Số lần cố dán"
  },
  {
   "name": "copy_attempts",
   "modality": "browser",
   "scale": 1,
   "source": "client",
   "desc": "Số lần cố sao chép/cắt đề"
  },
  {
   "name": "drop_attempts",
   "modality": "browser",
   "scale": 1,
   "source": "client",
   "desc": "Số lần cố kéo thả"
  },
  {
   "name": "contextmenu",
   "modality": "browser",
   "scale": 1,
   "source": "client",
   "desc": "Số lần mở menu chuột phải"
  },
  {
   "name": "chars_after_return",
   "modality": "browser",
   "scale": 20,
   "source": "client",
   "desc": "Số ký tự gõ trong 10 giây ngay sau khi quay lại tab"
  },
  {
   "name": "keys",
   "modality": "keystroke",
   "scale": 10,
   "source": "client",
   "desc": "Số phím (đã trừ phím do bộ gõ sinh)"
  },
  {
   "name": "dwell_rel",
   "modality": "keystroke",
   "scale": 1,
   "source": "client",
   "desc": "Thời gian giữ phím TB / mức khởi động"
  },
  {
   "name": "dwell_cv",
   "modality": "keystroke",
   "scale": 1,
   "source": "client",
   "desc": "Hệ số biến thiên thời gian giữ phím"
  },
  {
   "name": "flight_rel",
   "modality": "keystroke",
   "scale": 1,
   "source": "client",
   "desc": "Khoảng cách giữa hai phím TB / mức khởi động"
  },
  {
   "name": "flight_cv",
   "modality": "keystroke",
   "scale": 1,
   "source": "client",
   "desc": "Hệ số biến thiên khoảng cách giữa hai phím"
  },
  {
   "name": "burst_ratio",
   "modality": "keystroke",
   "scale": 1,
   "source": "client",
   "desc": "Tỉ lệ khoảng cách < 150 ms"
  },
  {
   "name": "pause_ratio",
   "modality": "keystroke",
   "scale": 1,
   "source": "client",
   "desc": "Tỉ lệ khoảng cách > 2 giây"
  },
  {
   "name": "backspace_ratio",
   "modality": "keystroke",
   "scale": 0.3,
   "source": "client",
   "desc": "Tỉ lệ phím xóa/sửa"
  },
  {
   "name": "compositions",
   "modality": "keystroke",
   "scale": 3,
   "source": "client",
   "desc": "Số lần bộ gõ IME ghép chữ"
  },
  {
   "name": "chars_per_sec",
   "modality": "keystroke",
   "scale": 5,
   "source": "client",
   "desc": "Ký tự/giây"
  },
  {
   "name": "question_switches",
   "modality": "answer",
   "scale": 1,
   "source": "client",
   "desc": "Số lần chuyển câu"
  },
  {
   "name": "answer_changes",
   "modality": "answer",
   "scale": 1,
   "source": "client",
   "desc": "Số lần đổi đáp án"
  },
  {
   "name": "time_on_question",
   "modality": "answer",
   "scale": 60,
   "source": "client",
   "desc": "Số giây đã ở câu hiện tại"
  },
  {
   "name": "progress",
   "modality": "answer",
   "scale": 1,
   "source": "client",
   "desc": "Tỉ lệ câu đã trả lời"
  },
  {
   "name": "time_vs_expected",
   "modality": "answer",
   "scale": 2,
   "source": "client",
   "desc": "(thời gian trả lời - kỳ vọng theo độ khó) / độ lệch, âm = nhanh bất thường"
  },
  {
   "name": "lag_ms",
   "modality": "system",
   "scale": 200,
   "source": "client",
   "desc": "Độ trễ khung hình giao diện làm bài (watchdog)"
  },
  {
   "name": "offline",
   "modality": "system",
   "scale": 1,
   "source": "client",
   "desc": "Mất kết nối trong cửa sổ (0/1, không coi là gian lận)"
  },
  {
   "name": "monitor_load",
   "modality": "system",
   "scale": 0.2,
   "source": "client",
   "desc": "Tỉ lệ thời gian CPU dành cho giám sát"
  }
 ],
 "model_input": "concat(clip(x/scale,-5,5) cho từng đặc trưng, mask theo phương thức 0/1) -> 54 + 8 = 62 chiều",
 "privacy": {
  "keystroke_laplace_epsilon": 2.0,
  "keystroke_quantize": 0.05,
  "warmup_seconds": 60
 }
}

In [ ]:
%%writefile eg_common.py
"""Mã dùng chung cho các notebook ExamGuard (được nhúng vào notebook bằng %%writefile để chạy độc lập trên Colab).

Các hàm đặc trưng ở đây là bản Python của frontend/src/monitor/features.ts — PHẢI giữ khớp công thức,
vì mô hình huấn luyện ở notebook sẽ chạy trên đặc trưng do trình duyệt thí sinh tính.
"""
from __future__ import annotations

import json
import math
from pathlib import Path

import numpy as np

# ----------------------------------------------------------------------------- schema
_HERE = Path(__file__).resolve().parent
SCHEMA_PATH = next(p for p in (_HERE / "feature_schema.json", _HERE.parent / "shared" / "feature_schema.json") if p.exists())
SCHEMA = json.loads(SCHEMA_PATH.read_text(encoding="utf-8"))
FEATURES = [f["name"] for f in SCHEMA["features"]]
MODALITIES = SCHEMA["modalities"]
SCALES = np.array([float(f["scale"]) for f in SCHEMA["features"]], np.float32)
MOD_OF = {f["name"]: f["modality"] for f in SCHEMA["features"]}
MOD_SLICES = {}
for _m in MODALITIES:
    _idx = [i for i, f in enumerate(SCHEMA["features"]) if f["modality"] == _m]
    MOD_SLICES[_m] = slice(_idx[0], _idx[-1] + 1)
DIM = len(FEATURES) + len(MODALITIES)
WINDOW_S = SCHEMA["window_seconds"]


def model_input(values: dict, masks: dict) -> np.ndarray:
    """Giống backend FeatureSpec.model_input: clip(x/scale) * mask + mask."""
    x = np.array([float(values.get(n, 0.0) or 0.0) for n in FEATURES], np.float32)
    x = np.clip(x / SCALES, -5, 5)
    m = np.array([1.0 if masks.get(mod) else 0.0 for mod in MODALITIES], np.float32)
    for k, mod in enumerate(MODALITIES):
        x[MOD_SLICES[mod]] *= m[k]
    return np.concatenate([x, m]).astype(np.float32)


# ----------------------------------------------------------------------------- khuôn mặt
def head_pose(kps) -> tuple[float, float]:
    """kps: 6 điểm BlazeFace chuẩn hóa [(x,y)...]: mắt phải, mắt trái, mũi, miệng, tai phải, tai trái."""
    if kps is None or len(kps) < 4:
        return 0.0, 0.0
    re, le, nose, mouth = [np.asarray(k[:2], float) for k in kps[:4]]
    mid = (re + le) / 2
    eye = max(float(np.hypot(*(le - re))), 1e-6)
    roll = math.atan2(le[1] - re[1], le[0] - re[0])
    c, s = math.cos(-roll), math.sin(-roll)

    def rot(p):
        d = p - mid
        return np.array([d[0] * c - d[1] * s, d[0] * s + d[1] * c])

    n, m = rot(nose), rot(mouth)
    yaw = math.degrees(math.atan(n[0] / eye / 0.6))
    pitch = (n[1] / max(m[1], 1e-6) - 0.55) * 120
    return float(np.clip(yaw, -90, 90)), float(np.clip(pitch, -60, 60))


def face_window(samples: list[dict]) -> dict:
    """samples: [{count, yaw, pitch, cx, cy, size, conf}] trong một cửa sổ 2 giây."""
    n = len(samples) or 1
    pres = [s for s in samples if s["count"] > 0]
    yaws = [s["yaw"] for s in pres]
    pit = [s["pitch"] for s in pres]
    shift = sum(math.hypot(pres[i]["cx"] - pres[i - 1]["cx"], pres[i]["cy"] - pres[i - 1]["cy"]) for i in range(1, len(pres)))
    sd = lambda a: float(np.std(a, ddof=1)) if len(a) > 1 else 0.0  # noqa: E731
    return {
        "face_count_mean": float(np.mean([s["count"] for s in samples])) if samples else 0.0,
        "face_count_max": max([s["count"] for s in samples], default=0),
        "face_missing_ratio": (len(samples) - len(pres)) / n if samples else 0.0,
        "yaw_mean": float(np.mean(yaws)) if yaws else 0.0, "yaw_std": sd(yaws),
        "pitch_mean": float(np.mean(pit)) if pit else 0.0, "pitch_std": sd(pit),
        "head_away_ratio": (sum(1 for s in pres if abs(s["yaw"]) > 30 or abs(s["pitch"]) > 25) / len(pres)) if pres else 0.0,
        "box_shift": shift,
        "face_size": float(np.mean([s["size"] for s in pres])) if pres else 0.0,
        "face_conf": float(np.mean([s["conf"] for s in pres])) if pres else 0.0,
    }


# ----------------------------------------------------------------------------- nhịp gõ
def key_timings(events: list[tuple]) -> tuple[list, list, int, int]:
    """events: [(t_ms, 'down'|'up', code, ime_bool)] -> dwell, flight, số phím, số phím sửa."""
    down, dwell, flight = {}, [], []
    last, keys, edits = None, 0, 0
    for t, typ, code, ime in events:
        if ime:
            continue
        if typ == "down":
            if code in down:
                continue
            down[code] = t
            keys += 1
            if code in ("Backspace", "Delete"):
                edits += 1
            if last is not None:
                flight.append(t - last)
            last = t
        else:
            t0 = down.pop(code, None)
            if t0 is not None:
                dwell.append(t - t0)
    return [d for d in dwell if 0 < d < 2000], [f for f in flight if 0 < f < 30000], keys, edits


def keystroke_window(events, compositions: int, inserted_chars: int, seconds: float, base: dict | None) -> dict:
    dwell, flight, keys, edits = key_timings(events)
    fl = [f for f in flight if f < 5000]
    dm = float(np.mean(dwell)) if dwell else 0.0
    fm = float(np.mean(fl)) if fl else 0.0
    sd = lambda a: float(np.std(a, ddof=1)) if len(a) > 1 else 0.0  # noqa: E731
    return {
        "keys": keys,
        "dwell_rel": (dm / base["dwell"] if base and dm else (1.0 if dm else 0.0)),
        "dwell_cv": sd(dwell) / dm if dm else 0.0,
        "flight_rel": (fm / base["flight"] if base and fm else (1.0 if fm else 0.0)),
        "flight_cv": sd(fl) / fm if fm else 0.0,
        "burst_ratio": sum(f < 150 for f in flight) / len(flight) if flight else 0.0,
        "pause_ratio": sum(f > 2000 for f in flight) / len(flight) if flight else 0.0,
        "backspace_ratio": edits / keys if keys else 0.0,
        "compositions": compositions,
        "chars_per_sec": inserted_chars / max(seconds, 1e-3),
    }


PRIVATE_KEYSTROKE = ["dwell_rel", "dwell_cv", "flight_rel", "flight_cv", "burst_ratio", "pause_ratio"]


def privatize(v: dict, epsilon: float, step: float = 0.05, rng: np.random.Generator | None = None) -> dict:
    """Lượng tử hóa + nhiễu Laplace(b = step/ε) như trên máy thí sinh. ε = inf -> chỉ lượng tử hóa."""
    rng = rng or np.random.default_rng()
    out = dict(v)
    for k in PRIVATE_KEYSTROKE:
        if k in out:
            noise = 0.0 if not np.isfinite(epsilon) else rng.laplace(0, step / max(epsilon, 1e-6))
            out[k] = max(0.0, round((out[k] + noise) / step) * step)
    return out


_NON_CHAR = {"backspace", "delete", "shift", "shiftleft", "shiftright", "control", "controlleft", "controlright", "ctrl",
             "alt", "altleft", "altright", "meta", "metaleft", "metaright", "tab", "enter", "capslock", "escape", "esc",
             "arrowup", "arrowdown", "arrowleft", "arrowright", "home", "end", "pageup", "pagedown", "insert"}


def is_char_key(code) -> bool:
    c = str(code).lower()
    return c not in _NON_CHAR and not c.startswith(("f1", "f2", "f3", "f4", "f5", "f6", "f7", "f8", "f9"))


def session_keystroke_windows(events: list[tuple], warmup_s: float = 60, epsilon: float = float("inf"),
                              window_s: float = WINDOW_S, rng=None) -> list[dict]:
    """Cắt chuỗi phím (t_ms tuyệt đối) thành cửa sổ 2 giây với chuẩn hóa theo đoạn khởi động, như trình duyệt."""
    if not events:
        return []
    events = sorted(events, key=lambda e: e[0])
    t0 = events[0][0]
    base, warm_d, warm_f = None, [], []
    out, cur, k = [], [], 0
    end = events[-1][0]
    w_start = t0
    i = 0
    while w_start <= end:
        w_end = w_start + window_s * 1000
        cur = []
        while i < len(events) and events[i][0] < w_end:
            cur.append(events[i])
            i += 1
        if base is None:
            d, f, _, _ = key_timings(cur)
            warm_d += d
            warm_f += [x for x in f if x < 5000]
            if (w_end - t0) / 1000 >= warmup_s and len(warm_d) >= 30 and len(warm_f) >= 30:
                base = {"dwell": float(np.median(warm_d)), "flight": float(np.median(warm_f))}
        chars = sum(1 for e in cur if e[1] == "down" and not e[3] and is_char_key(e[2]))
        w = keystroke_window(cur, 0, chars, window_s, base)
        w = privatize(w, epsilon, rng=rng) if epsilon is not None else w
        d, f, _, _ = key_timings(cur)
        # thời gian tuyệt đối (ms): chỉ dùng để đánh giá rủi ro tái nhận dạng ở notebook 04 — trình duyệt KHÔNG gửi
        w["_dwell_ms"] = float(np.mean(d)) if d else 0.0
        w["_flight_ms"] = float(np.mean([x for x in f if x < 5000])) if f else 0.0
        w["_dwell_ms_std"] = float(np.std(d)) if len(d) > 1 else 0.0
        out.append(w)
        w_start = w_end
        k += 1
    return out


# ----------------------------------------------------------------------------- âm thanh
def audio_window(speech_flags: list[bool], rms: list[float]) -> dict:
    onsets = sum(1 for i, s in enumerate(speech_flags) if s and (i == 0 or not speech_flags[i - 1]))
    r = float(np.mean(rms)) if rms else 0.0
    return {"speech_ratio": float(np.mean(speech_flags)) if speech_flags else 0.0, "speech_segments": onsets,
            "audio_energy": float(np.clip((20 * math.log10(r) + 60) / 60, 0, 1)) if r > 0 else 0.0}


def time_vs_expected(seconds: float, expected: float) -> float:
    return float(np.clip((seconds - expected) / (0.5 * expected), -5, 5)) if expected else 0.0


# ----------------------------------------------------------------------------- baseline luật ngưỡng (khớp backend)
RULES = [
    ("MULTI_FACE", "face", 0.60, lambda v: v["face_count_max"] >= 2),
    ("NO_FACE", "face", 0.25, lambda v: v["face_missing_ratio"] >= 0.99),
    ("HEAD_AWAY", "face", 0.15, lambda v: v["head_away_ratio"] >= 0.5),
    ("ID_MISMATCH", "identity", 0.70, lambda v: v["id_mismatch"] >= 1),
    ("PHONE", "object", 0.60, lambda v: v["obj_phone"] >= 0.5),
    ("PERSON_EXTRA", "object", 0.50, lambda v: v["obj_person_extra"] >= 0.5),
    ("HEADPHONE", "object", 0.35, lambda v: v["obj_headphone"] >= 0.5),
    ("BOOK", "object", 0.30, lambda v: v["obj_book"] >= 0.5),
    ("SCREEN", "object", 0.30, lambda v: v["obj_screen"] >= 0.5),
    ("SPEECH", "audio", 0.20, lambda v: v["speech_ratio"] >= 0.3),
    ("TAB_AWAY", "browser", 0.40, lambda v: v["hidden_sec"] > 0 or v["tab_switches"] > 0),
    ("BLUR", "browser", 0.20, lambda v: v["blur_sec"] >= 1),
    ("FULLSCREEN_EXIT", "browser", 0.20, lambda v: v["fullscreen_exits"] > 0 or v["not_fullscreen_sec"] >= 2),
    ("WINDOW_SMALL", "browser", 0.15, lambda v: v["window_small"] >= 1),
    ("EXTENDED_SCREEN", "browser", 0.20, lambda v: v["screen_extended"] >= 1),
    ("PASTE", "browser", 0.40, lambda v: v["paste_attempts"] > 0 or v["drop_attempts"] > 0),
    ("COPY", "browser", 0.25, lambda v: v["copy_attempts"] > 0),
    ("TYPE_AFTER_RETURN", "browser", 0.40, lambda v: v["chars_after_return"] >= 20),
    ("TRANSCRIBE_RHYTHM", "keystroke", 0.20, lambda v: v["keys"] >= 10 and v["pause_ratio"] < 0.02
     and v["backspace_ratio"] < 0.02 and v["burst_ratio"] > 0.6 and v["flight_cv"] < 0.5),
    ("FAST_ANSWER", "answer", 0.15, lambda v: v["time_vs_expected"] <= -2.5),
]


def rule_score(values: dict, masks: dict) -> float:
    v = {n: float(values.get(n, 0.0)) for n in FEATURES}
    p = 1.0
    for _, mod, w, test in RULES:
        if masks.get(mod) and test(v):
            p *= 1 - w
    return 1 - p


def rule_score_matrix(X: np.ndarray) -> np.ndarray:
    """X: [N, DIM] (đầu vào mô hình) -> điểm luật. Giải chuẩn hóa bằng scale."""
    out = np.zeros(len(X), np.float32)
    for i, x in enumerate(X):
        vals = dict(zip(FEATURES, (x[:len(FEATURES)] * SCALES).tolist()))
        masks = {m: x[len(FEATURES) + k] > 0.5 for k, m in enumerate(MODALITIES)}
        out[i] = rule_score(vals, masks)
    return out


# ----------------------------------------------------------------------------- đánh giá
def recall_at_fpr(y: np.ndarray, s: np.ndarray, fpr: float) -> float:
    y, s = np.asarray(y).astype(bool), np.asarray(s, float)
    neg = np.sort(s[~y])[::-1]
    if not len(neg) or not y.any():
        return float("nan")
    k = int(np.floor(fpr * len(neg)))
    thr = neg[k] if k < len(neg) else -np.inf
    return float((s[y] > thr).mean())


def summarize(y, s) -> dict:
    from sklearn.metrics import average_precision_score, roc_auc_score

    y = np.asarray(y).astype(int)
    if y.min() == y.max():
        return {"n": len(y), "pos": int(y.sum())}
    return {"n": len(y), "pos": int(y.sum()), "auc": round(roc_auc_score(y, s), 4), "ap": round(average_precision_score(y, s), 4),
            "recall@fpr1%": round(recall_at_fpr(y, s, 0.01), 4), "recall@fpr5%": round(recall_at_fpr(y, s, 0.05), 4)}


# ----------------------------------------------------------------------------- mô hình hợp nhất (PyTorch)
def build_fusion_model(hidden: int = 64, d: int = 32, dropout: float = 0.1):
    import torch
    from torch import nn

    class FusionGRU(nn.Module):
        """Encoder riêng từng phương thức -> cổng (gated fusion, che phương thức thiếu) -> GRU nhân quả -> xác suất.

        Một bước = một cửa sổ 2 giây; trạng thái ẩn h mang thông tin quá khứ (streaming)."""

        def __init__(self):
            super().__init__()
            self.slices = [MOD_SLICES[m] for m in MODALITIES]
            self.enc = nn.ModuleList([nn.Sequential(nn.Linear(s.stop - s.start, d), nn.GELU(), nn.Linear(d, d))
                                      for s in self.slices])
            self.gate = nn.Linear(len(MODALITIES) * d, len(MODALITIES))
            self.drop = nn.Dropout(dropout)
            self.gru = nn.GRUCell(d, hidden)
            self.head = nn.Sequential(nn.Linear(hidden + d, hidden), nn.GELU(), nn.Linear(hidden, 1))
            self.hidden = hidden
            self.recon = nn.Linear(hidden + d, len(FEATURES))  # cho tiền huấn luyện tự giám sát

        def fuse(self, x):
            feats = x[..., :len(FEATURES)]
            mask = x[..., len(FEATURES):]
            z = torch.stack([enc(feats[..., s]) for enc, s in zip(self.enc, self.slices)], dim=-2)  # [..., M, d]
            logits = self.gate(z.flatten(-2)) + (mask - 1) * 1e4  # phương thức thiếu -> cổng = 0
            g = torch.softmax(logits, dim=-1)
            g = g * (mask.sum(-1, keepdim=True) > 0)
            fused = (g.unsqueeze(-1) * z).sum(-2)
            return self.drop(fused), g

        def step(self, x, h):
            fused, g = self.fuse(x)
            h2 = self.gru(fused, h)
            out = torch.cat([h2, fused], -1)
            return torch.sigmoid(self.head(out)).squeeze(-1), h2, g

        def forward(self, xs):  # xs: [B, T, D] -> logits [B, T], gates [B, T, M], recon [B, T, F]
            B, T, _ = xs.shape
            h = xs.new_zeros(B, self.hidden)
            logits, gates, recs = [], [], []
            for t in range(T):
                fused, g = self.fuse(xs[:, t])
                h = self.gru(fused, h)
                out = torch.cat([h, fused], -1)
                logits.append(self.head(out).squeeze(-1))
                gates.append(g)
                recs.append(self.recon(out))
            return torch.stack(logits, 1), torch.stack(gates, 1), torch.stack(recs, 1)

    return FusionGRU()


def export_fusion_onnx(model, path: str) -> None:
    """Xuất MỘT BƯỚC streaming: x[1,D], h[1,H] -> prob[1], h_out[1,H], gates[1,M] (đúng tên backend đọc)."""
    import torch

    class Step(torch.nn.Module):
        def __init__(self, m):
            super().__init__()
            self.m = m

        def forward(self, x, h):
            return self.m.step(x, h)

    model.eval()
    step = Step(model)
    x = torch.zeros(1, DIM)
    h = torch.zeros(1, model.hidden)
    torch.onnx.export(step, (x, h), path, input_names=["x", "h"], output_names=["prob", "h_out", "gates"],
                      opset_version=17, dynamo=False)
    import onnx

    m = onnx.load(path)
    m.ir_version = min(m.ir_version, 9)  # onnxruntime trên server có thể là bản cũ
    m.metadata_props.add(key="schema_version", value=str(SCHEMA["version"]))
    m.metadata_props.add(key="features", value=",".join(FEATURES))
    onnx.save(m, path)


def check_onnx_parity(model, path: str, xs: np.ndarray, atol: float = 1e-4) -> float:
    """Chạy chuỗi bằng ONNX từng bước, so với PyTorch chạy cả chuỗi."""
    import onnxruntime as ort
    import torch

    sess = ort.InferenceSession(path, providers=["CPUExecutionProvider"])
    model.eval()
    with torch.no_grad():
        ref = torch.sigmoid(model(torch.from_numpy(xs[None]).float())[0])[0].numpy()
    h = np.zeros((1, model.hidden), np.float32)
    got = []
    for x in xs:
        p, h, _ = sess.run(None, {"x": x[None].astype(np.float32), "h": h})
        got.append(float(np.asarray(p).reshape(-1)[0]))
    err = float(np.max(np.abs(np.array(got) - ref)))
    assert err < atol, err
    return err


# ----------------------------------------------------------------------------- phiên thi tổng hợp
def ks_window(kind: str, rng, ks_bona=None, ks_copy=None) -> np.ndarray:
    pool = ks_bona if kind == "bona" else ks_copy
    if pool is not None and len(pool):
        return pool[rng.integers(len(pool))]
    # mô phỏng: [keys, dwell_rel, dwell_cv, flight_rel, flight_cv, burst, pause, backspace, compositions, cps]
    if kind == "bona":
        k = rng.poisson(8)
        return np.array([k, rng.normal(1, .12), rng.uniform(.3, .7), rng.normal(1, .25), rng.uniform(.6, 1.4),
                         rng.uniform(.1, .4), rng.uniform(0, .15), rng.uniform(.03, .15), rng.poisson(1), k / 2], np.float32)
    k = rng.poisson(14)
    return np.array([k, rng.normal(.95, .08), rng.uniform(.2, .4), rng.normal(.8, .1), rng.uniform(.3, .7),
                     rng.uniform(.4, .8), rng.uniform(0, .03), rng.uniform(0, .03), rng.poisson(1), k / 2], np.float32)


EPISODES = ["tab_lookup", "phone", "second_person", "whisper", "paste", "second_screen", "llm_transcribe"]


def synth_session(sid: int, T: int, cheat: bool, tier: str, rng, ks_bona=None, ks_copy=None) -> dict:
    """Một phiên thi tổng hợp: nền bình thường có nhiễu thực tế + 1–3 đoạn gian lận nếu cheat."""
    FI = {n: i for i, n in enumerate(FEATURES)}
    ks = lambda kind: ks_window(kind, rng, ks_bona, ks_copy)  # noqa: E731
    V = np.zeros((T, len(FEATURES)), np.float32)
    y = np.zeros(T, np.int64)
    f = lambda name: FI[name]  # noqa: E731
    # --- nền bình thường
    miss_p = {"low": .12, "mid": .04, "high": .02}[tier]
    present = rng.random(T) > miss_p
    V[:, f("face_count_mean")] = present
    V[:, f("face_count_max")] = present
    V[:, f("face_missing_ratio")] = ~present
    V[:, f("yaw_mean")] = rng.normal(0, 8, T) * present
    V[:, f("yaw_std")] = np.abs(rng.normal(3, 2, T)) * present
    V[:, f("pitch_mean")] = rng.normal(5, 6, T) * present
    V[:, f("pitch_std")] = np.abs(rng.normal(2, 1.5, T)) * present
    glance = rng.random(T) < .02
    V[glance, f("yaw_mean")] = rng.choice([-1, 1], glance.sum()) * rng.uniform(30, 50, glance.sum())
    V[:, f("head_away_ratio")] = (np.abs(V[:, f("yaw_mean")]) > 30).astype(np.float32)
    V[:, f("box_shift")] = np.abs(rng.normal(.01, .01, T))
    V[:, f("face_size")] = rng.uniform(.06, .14) * present
    V[:, f("face_conf")] = rng.uniform(.8, .97, T) * present
    V[:, f("id_score")] = rng.uniform(.55, .8)
    V[:, f("obj_age")] = rng.uniform(0, 10, T)
    cough = rng.random(T) < .01
    V[:, f("speech_ratio")] = cough * rng.uniform(.1, .3, T)
    V[:, f("speech_segments")] = cough
    V[:, f("audio_energy")] = rng.uniform(.05, .2, T) + cough * .3
    notif = rng.random(T) < .004  # thông báo hệ thống làm mất focus ngắn
    V[notif, f("blur_sec")] = rng.uniform(.3, 1.5, notif.sum())
    typing = rng.random(T) < .35
    for t in np.where(typing)[0]:
        V[t, MOD_SLICES["keystroke"]] = ks("bona")
    V[:, f("question_switches")] = rng.random(T) < .03
    V[:, f("answer_changes")] = rng.random(T) < .05
    V[:, f("time_on_question")] = rng.uniform(0, 120, T)
    V[:, f("progress")] = np.linspace(0, 1, T)
    V[:, f("time_vs_expected")] = np.where(rng.random(T) < .05, rng.normal(0, 1, T), 0)
    lag = {"low": 120, "mid": 40, "high": 15}[tier]
    V[:, f("lag_ms")] = np.abs(rng.normal(lag, lag / 2, T))
    off = rng.random(T) < .003
    V[:, f("offline")] = off
    V[:, f("monitor_load")] = {"low": .12, "mid": .05, "high": .02}[tier]
    # --- phương thức có/không theo phiên
    masks = {"face": 1, "identity": int(rng.random() < .8), "object": int(rng.random() < .7), "audio": int(rng.random() < .85),
             "browser": 1, "keystroke": 1, "answer": 1, "system": 1}
    # --- đoạn gian lận
    kinds = []
    if cheat:
        for _ in range(rng.integers(1, 4)):
            kind = EPISODES[rng.integers(len(EPISODES))]
            L = int(rng.integers(5, 45))
            s0 = int(rng.integers(10, max(11, T - L - 1)))
            sl = slice(s0, s0 + L)
            y[sl] = 1
            kinds.append(kind)
            if kind == "tab_lookup":
                V[sl, f("hidden_sec")] = 2.0
                V[sl, f("tab_switches")] = 0
                V[s0, f("tab_switches")] = 1
                V[sl, f("face_count_mean")] = 1
                after = slice(s0 + L, min(T, s0 + L + 5))
                V[after, f("chars_after_return")] = rng.uniform(20, 80)
                for t in range(after.start, after.stop):
                    V[t, MOD_SLICES["keystroke"]] = ks("copy")
                y[after] = 1
            elif kind == "phone":
                V[sl, f("pitch_mean")] = rng.uniform(25, 45)
                V[sl, f("head_away_ratio")] = 1
                V[sl, f("obj_phone")] = rng.uniform(.4, .9, L) * (rng.random(L) < .6)
            elif kind == "second_person":
                V[sl, f("face_count_max")] = (rng.random(L) < .5) + 1
                V[sl, f("obj_person_extra")] = rng.uniform(.3, .9, L) * (rng.random(L) < .6)
                V[sl, f("speech_ratio")] = rng.uniform(.2, .7, L)
                V[sl, f("speech_segments")] = rng.integers(1, 3, L)
            elif kind == "whisper":
                V[sl, f("speech_ratio")] = rng.uniform(.1, .5, L)
                V[sl, f("audio_energy")] = rng.uniform(.15, .35, L)
            elif kind == "paste":
                V[s0, f("paste_attempts")] = rng.integers(1, 3)
                y[sl] = 0
                y[s0:s0 + 3] = 1
            elif kind == "second_screen":
                V[sl, f("screen_extended")] = 1
                V[sl, f("yaw_mean")] = rng.choice([-1, 1]) * rng.uniform(30, 55, L)
                V[sl, f("head_away_ratio")] = 1
                V[sl, f("obj_screen")] = rng.uniform(.3, .8, L) * (rng.random(L) < .4)
            elif kind == "llm_transcribe":
                for t in range(sl.start, sl.stop):
                    V[t, MOD_SLICES["keystroke"]] = ks("copy")
                V[sl, f("time_vs_expected")] = rng.uniform(-4, -2)
    X = np.stack([model_input(dict(zip(FEATURES, row)), masks) for row in V])
    return {"id": f"syn:{sid}", "X": X, "y": y, "tier": tier, "source": "synthetic", "kinds": kinds}




def make_synthetic_sessions(n: int, T: int, rng, cheat_rate: float = 0.5, tiers=("low", "mid", "high"),
                            tier_p=(0.3, 0.5, 0.2), ks_bona=None, ks_copy=None) -> list[dict]:
    return [synth_session(i, T, cheat=rng.random() < cheat_rate, tier=rng.choice(list(tiers), p=list(tier_p)), rng=rng,
                          ks_bona=ks_bona, ks_copy=ks_copy) for i in range(n)]

In [ ]:
import json
import math
import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from eg_common import (DIM, EPISODES, FEATURES, MOD_OF, MOD_SLICES, MODALITIES, SCALES, WINDOW_S, build_fusion_model,
                       check_onnx_parity, export_fusion_onnx, make_synthetic_sessions, model_input, rule_score_matrix,
                       summarize)

OUT = Path("/content/examguard")
try:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT = Path("/content/drive/MyDrive/examguard")
except Exception as e:
    print("Không mount được Drive:", e)
OUT.mkdir(parents=True, exist_ok=True)
DATA = Path("/content/data") if Path("/content").exists() else Path("data")
DATA.mkdir(parents=True, exist_ok=True)
DEV = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 0
rng = np.random.default_rng(SEED)
torch.manual_seed(SEED)
random.seed(SEED)
FI = {n: i for i, n in enumerate(FEATURES)}
print("Thiết bị:", DEV, "| đầu vào mô hình:", DIM, "chiều")

## 1. OEP: tải từ Kaggle một lần, lưu trên Google Drive
- **Lần đầu:** tải `kaggle.json` (Kaggle → Settings → API → Create New Token) lên khi được hỏi. Ô này tải bộ OEP (~12 GB nén),
  chỉ giữ phần cần dùng (video webcam `*1.avi`, âm thanh `*.wav`, nhãn `gt.txt`, ~4 GB) và chép vào
  `MyDrive/examguard_data/oep/`. `kaggle.json` cũng được lưu vào `MyDrive/examguard_data/` để lần sau khỏi tải lên lại.
- **Các lần sau:** chép từ Drive về ổ máy Colab (nhanh hơn tải lại), không cần Kaggle nữa.
- Cần ~5 GB trống trên Drive. Video đeo kính (`*2.avi`, ~8 GB) không dùng nên không lưu; đặt `KEEP_WEARCAM = True` nếu cần.
- Đặc trưng đã trích được lưu ở `MyDrive/examguard/oep_features.npz`; khi file này có rồi thì không cần video nữa.
  Xóa file đó nếu muốn trích lại (ví dụ sau khi huấn luyện lại YOLO ở notebook 02).

Không có `kaggle.json` thì đặt `USE_OEP = False` (vẫn huấn luyện được trên phiên tổng hợp, nhưng kết quả **không** đủ cho bài báo).

In [ ]:
import shutil
import zipfile

USE_OEP = True
KEEP_WEARCAM = False
DRIVE_DATA = OUT.parent / "examguard_data"  # MyDrive/examguard_data khi đã mount Drive
DRIVE_OEP = DRIVE_DATA / "oep"
DRIVE_OK = str(OUT).startswith("/content/drive")
OEP_DIR = DATA / "oep"


def _keep(name: str) -> bool:
    base = name.rsplit("/", 1)[-1].lower()
    if base in ("gt.txt", "read_me.txt") or base.endswith(".wav") or base.endswith("1.avi"):
        return True
    return KEEP_WEARCAM and base.endswith(".avi")


def _kaggle_ready() -> bool:
    kj = Path.home() / ".kaggle" / "kaggle.json"
    kj.parent.mkdir(exist_ok=True)
    saved = DRIVE_DATA / "kaggle.json"
    if not kj.exists() and saved.exists():
        shutil.copy(saved, kj)
    if not kj.exists():
        try:
            from google.colab import files
            print("Chọn file kaggle.json")
            up = files.upload()
            kj.write_bytes(up["kaggle.json"])
        except Exception as e:
            print("Không có kaggle.json:", e)
            return False
    kj.chmod(0o600)
    if DRIVE_OK and not saved.exists():
        DRIVE_DATA.mkdir(parents=True, exist_ok=True)
        shutil.copy(kj, saved)
        print("Đã lưu kaggle.json vào", saved)
    return True


def _drive_has_oep() -> bool:
    return (DRIVE_OEP / "_done").exists()


if USE_OEP and not any(OEP_DIR.rglob("gt.txt")):
    if DRIVE_OK and _drive_has_oep():
        print("Chép OEP từ Drive về máy Colab...")
        shutil.copytree(DRIVE_OEP, OEP_DIR, dirs_exist_ok=True)
    elif _kaggle_ready():
        zpath = DATA / "oep-dataset.zip"
        if not zpath.exists():
            print("Tải OEP từ Kaggle (~12 GB, khoảng 5–15 phút)...")
            os.system(f"kaggle datasets download -d raajanwankhade/oep-dataset -p {DATA} -q")
        if zpath.exists():
            with zipfile.ZipFile(zpath) as z:
                members = [m for m in z.namelist() if not m.endswith("/") and _keep(m)]
                print("Giải nén", len(members), "file cần dùng...")
                for m in members:
                    rel = Path(*Path(m).parts[1:]) if len(Path(m).parts) > 1 else Path(m)  # bỏ thư mục "OEP database/"
                    dst = OEP_DIR / rel
                    dst.parent.mkdir(parents=True, exist_ok=True)
                    with z.open(m) as src, open(dst, "wb") as f:
                        shutil.copyfileobj(src, f, 1 << 24)
            zpath.unlink()  # giải phóng ổ Colab
            if DRIVE_OK:
                print("Chép vào Drive:", DRIVE_OEP, "(vài phút)...")
                shutil.copytree(OEP_DIR, DRIVE_OEP, dirs_exist_ok=True)
                (DRIVE_OEP / "_done").write_text("ok")
            else:
                print("Không có Drive: dữ liệu chỉ nằm trên máy Colab, sẽ mất khi ngắt phiên.")
        else:
            print("Tải thất bại. Kiểm tra lại kaggle.json (token có thể đã hết hạn: tạo token mới trên Kaggle).")
subjects = sorted({p.parent for p in OEP_DIR.rglob("gt.txt")}) if USE_OEP else []
USE_OEP = USE_OEP and len(subjects) > 0
print("OEP:", len(subjects), "người", "(TẮT — chỉ dùng dữ liệu tổng hợp)" if not USE_OEP else "")
if subjects:
    print("Ví dụ gt.txt:", (subjects[0] / "gt.txt").read_text()[:200])

### Đọc nhãn `gt.txt`
Mỗi dòng: `bắt_đầu kết_thúc loại`. Thời gian dạng `MMSS` (vd `0123` = 1 phút 23 giây) hoặc `mm:ss` hoặc giây.
**Kiểm tra lại** với vài dòng in ở trên trước khi chạy tiếp.

In [ ]:
def parse_time(tok: str) -> float:
    tok = tok.strip()
    if ":" in tok:
        parts = [float(x) for x in tok.split(":")]
        return sum(v * 60 ** (len(parts) - 1 - i) for i, v in enumerate(parts))
    if tok.isdigit() and len(tok) in (3, 4):
        return int(tok[:-2]) * 60 + int(tok[-2:])
    return float(tok)


def parse_gt(path: Path) -> list[tuple[float, float, str]]:
    out = []
    for line in path.read_text(errors="ignore").splitlines():
        toks = line.replace(",", " ").split()
        if len(toks) >= 2:
            try:
                a, b = parse_time(toks[0]), parse_time(toks[1])
            except ValueError:
                continue
            if b >= a:
                out.append((a, b, toks[2] if len(toks) > 2 else "?"))
    return out


if USE_OEP:
    for s in subjects[:3]:
        print(s.name, parse_gt(s / "gt.txt")[:4])

## 2. Trích đặc trưng OEP bằng đúng công thức của trình duyệt
- Khuôn mặt: MediaPipe BlazeFace short-range, 1 khung/giây → `face_window` (6 điểm mốc → yaw/pitch).
- Âm thanh: Silero VAD trên âm thanh video, khung 100 ms → `audio_window`.
- Vật thể: YOLO (notebook 02) trên khung 320 px đã làm mờ mặt, mỗi 5 giây; giữ kết quả 15 giây như server.
- Danh tính / trình duyệt / nhịp gõ / bài làm: OEP không có → mask = 0 (đúng tình huống "thiếu phương thức");
  ở mục 4 sẽ ghép thêm nền trình duyệt + bài làm bình thường để không lệch phân bố.

In [ ]:
import subprocess
import urllib.request

import cv2

from eg_common import audio_window, face_window, head_pose

BLAZE = DATA / "blaze_face_short_range.tflite"
if USE_OEP and not BLAZE.exists():
    urllib.request.urlretrieve("https://storage.googleapis.com/mediapipe-models/face_detector/blaze_face_short_range/float16/1/blaze_face_short_range.tflite", BLAZE)
YOLO_ONNX = OUT / "proctor_yolo.onnx"  # từ notebook 02 (không có thì bỏ qua vật thể)


def make_face_detector():
    import mediapipe as mp
    from mediapipe.tasks.python import BaseOptions, vision

    return vision.FaceDetector.create_from_options(vision.FaceDetectorOptions(
        base_options=BaseOptions(model_asset_path=str(BLAZE)), min_detection_confidence=0.5)), mp


def make_yolo():
    if not YOLO_ONNX.exists():
        return None
    import ast

    import onnxruntime as ort
    sess = ort.InferenceSession(str(YOLO_ONNX), providers=["CUDAExecutionProvider", "CPUExecutionProvider"])
    names = {int(k): v for k, v in ast.literal_eval(sess.get_modelmeta().custom_metadata_map["names"]).items()}
    return sess, names


GROUPS = {"obj_phone": ["cell phone"], "obj_book": ["book"], "obj_headphone": ["headphones"], "obj_screen": ["laptop", "tv"]}


def yolo_groups(yolo, img320) -> dict:
    sess, names = yolo
    h, w = img320.shape[:2]
    canvas = np.full((320, 320, 3), 114, np.uint8)
    canvas[:h, :w] = img320
    x = canvas[..., ::-1].transpose(2, 0, 1)[None].astype(np.float32) / 255
    o = sess.run(None, {sess.get_inputs()[0].name: x})[0][0]
    cls = o[4:]
    res = {g: float(max((cls[i].max() for i, n in names.items() if n in labs), default=0.0)) for g, labs in GROUPS.items()}
    pid = [i for i, n in names.items() if n == "person"]
    if pid:
        sc = np.sort(cls[pid[0]])[::-1]
        res["obj_person_extra"] = float(sc[1]) if len(sc) > 1 and sc[1] > 0.25 else 0.0  # gần đúng (chưa NMS)
    return res


def pixelate(img, dets, W, H):
    for d in dets:
        b = d.bounding_box
        cx, cy = b.origin_x + b.width / 2, b.origin_y + b.height / 2
        bw, bh = b.width * 1.5, b.height * 1.7
        x0, y0 = int(max(0, cx - bw / 2)), int(max(0, cy - bh / 2))
        x1, y1 = int(min(W, cx + bw / 2)), int(min(H, cy + bh / 2))
        if x1 - x0 > 2 and y1 - y0 > 2:
            img[y0:y1, x0:x1] = cv2.resize(cv2.resize(img[y0:y1, x0:x1], (6, 8)), (x1 - x0, y1 - y0), interpolation=cv2.INTER_NEAREST)
    return img


def vad_frames(video: Path) -> tuple[np.ndarray, np.ndarray]:
    """Cờ có tiếng nói + RMS mỗi 100 ms. OEP ghi micro thành file .wav riêng; không có thì lấy âm thanh trong video."""
    from silero_vad import get_speech_timestamps, load_silero_vad
    src = next(iter(sorted(video.parent.glob("*.wav"))), video)
    wav = DATA / (video.stem + "_16k.wav")
    subprocess.run(["ffmpeg", "-loglevel", "error", "-y", "-i", str(src), "-ac", "1", "-ar", "16000", str(wav)], check=False)
    if not wav.exists():
        return np.zeros(0, bool), np.zeros(0)
    import soundfile as sf
    audio, sr = sf.read(str(wav), dtype="float32")
    model = load_silero_vad()
    ts = get_speech_timestamps(torch.from_numpy(audio), model, sampling_rate=sr, return_seconds=True)
    n = int(len(audio) / sr * 10)
    flags = np.zeros(n, bool)
    for t in ts:
        flags[int(t["start"] * 10):int(t["end"] * 10)] = True
    rms = np.array([np.sqrt(np.mean(audio[i * 1600:(i + 1) * 1600] ** 2) + 1e-12) for i in range(n)])
    return flags, rms


def extract_oep(subject: Path) -> dict | None:
    vids = sorted(subject.glob("*1.avi")) or sorted(subject.glob("*.avi"))  # *1.avi = webcam
    if not vids:
        return None
    video = vids[0]
    det, mp = make_face_detector()
    yolo = make_yolo()
    cap = cv2.VideoCapture(str(video))
    fps = cap.get(cv2.CAP_PROP_FPS) or 30
    n_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    dur = n_frames / fps
    T = int(dur // WINDOW_S)
    face_samples = [[] for _ in range(T)]
    obj = {}  # nhóm -> (xác suất, thời điểm)
    obj_rows = [None] * T
    last_frame_t = -1e9
    for sec in range(int(dur)):
        cap.set(cv2.CAP_PROP_POS_MSEC, sec * 1000)
        ok, frame = cap.read()
        if not ok:
            continue
        H, W = frame.shape[:2]
        res = det.detect(mp.Image(image_format=mp.ImageFormat.SRGB, data=cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)))
        dets = sorted(res.detections, key=lambda d: -d.bounding_box.width * d.bounding_box.height)
        if dets:
            b = dets[0].bounding_box
            kps = [(k.x, k.y) for k in dets[0].keypoints]
            yaw, pitch = head_pose(kps)
            s = {"count": len(dets), "yaw": yaw, "pitch": pitch, "cx": (b.origin_x + b.width / 2) / W,
                 "cy": (b.origin_y + b.height / 2) / H, "size": b.width * b.height / (W * H), "conf": dets[0].categories[0].score}
        else:
            s = {"count": 0, "yaw": 0, "pitch": 0, "cx": 0, "cy": 0, "size": 0, "conf": 0}
        k = int(sec // WINDOW_S)
        if k < T:
            face_samples[k].append(s)
        if yolo and sec % 5 == 0:
            small = cv2.resize(pixelate(frame.copy(), dets, W, H), (320, int(320 * H / W)))
            for g, v in yolo_groups(yolo, small).items():
                obj[g] = (v, sec)
            last_frame_t = sec
        if k < T:
            obj_rows[k] = {**{g: (v if sec - t <= 15 else 0.0) for g, (v, t) in obj.items()},
                           "obj_age": min(sec - last_frame_t, 30)}
    cap.release()
    flags, rms = vad_frames(video)
    gt = parse_gt(subject / "gt.txt")
    X, y = [], []
    for k in range(T):
        v = face_window(face_samples[k]) if face_samples[k] else {}
        a0, a1 = int(k * WINDOW_S * 10), int((k + 1) * WINDOW_S * 10)
        has_audio = len(flags) >= a1
        if has_audio:
            v.update(audio_window(list(flags[a0:a1]), list(rms[a0:a1])))
        has_obj = yolo is not None and obj_rows[k] is not None
        if has_obj:
            v.update(obj_rows[k])
        masks = {"face": bool(face_samples[k]), "audio": has_audio, "object": has_obj}
        X.append(model_input(v, masks))
        t0, t1 = k * WINDOW_S, (k + 1) * WINDOW_S
        y.append(int(any(a < t1 and b > t0 for a, b, _ in gt)))
    return {"id": f"oep:{subject.name}", "X": np.stack(X), "y": np.array(y), "tier": "mid", "source": "oep"}


oep_sessions = []
if USE_OEP:
    cache = OUT / "oep_features.npz"
    if cache.exists():
        z = np.load(cache, allow_pickle=True)
        oep_sessions = list(z["sessions"])
    else:
        from tqdm.auto import tqdm
        for s in tqdm(subjects):
            r = extract_oep(s)
            if r:
                oep_sessions.append(r)
        np.savez_compressed(cache, sessions=np.array(oep_sessions, dtype=object))
    print(len(oep_sessions), "phiên OEP; tỉ lệ cửa sổ gian lận:",
          round(float(np.mean(np.concatenate([s["y"] for s in oep_sessions]))), 3))

## 3. Phiên thi tổng hợp
Mỗi phiên = chuỗi cửa sổ 2 giây. Hành vi bình thường có nhiễu thực tế (nhìn ra ngoài thoáng qua, thông báo làm mất focus,
ho, mất mạng, máy yếu bỏ khung). Phiên gian lận có 1–3 đoạn thuộc các kiểu dưới đây. Cửa sổ nhịp gõ lấy từ dữ liệu thật
(notebook 01: B cho bình thường; T/FT cho chép câu trả lời LLM) nếu có file `keystroke_windows.parquet`.

In [ ]:
KS = OUT / "keystroke_windows.parquet"
KFEATS = [f for f in FEATURES if MOD_OF[f] == "keystroke"]
if KS.exists():
    kw = pd.read_parquet(KS)
    KS_BONA = kw[kw.label == "B"][KFEATS].values.astype(np.float32)
    KS_COPY = kw[kw.label.isin(["T", "FT", "P", "FP"])][KFEATS].values.astype(np.float32)
    print("Nhịp gõ thật:", len(KS_BONA), "cửa sổ bình thường,", len(KS_COPY), "cửa sổ có hỗ trợ LLM")
else:
    KS_BONA = KS_COPY = None
    print("Không có keystroke_windows.parquet: dùng nhịp gõ mô phỏng (chạy notebook 01 để có dữ liệu thật).")


N_SYN, T_SYN = 600, 450  # 600 phiên × 15 phút
syn_sessions = make_synthetic_sessions(N_SYN, T_SYN, rng, ks_bona=KS_BONA, ks_copy=KS_COPY)
sessions = syn_sessions + oep_sessions
print(len(sessions), "phiên; tỉ lệ cửa sổ gian lận", round(float(np.mean(np.concatenate([s["y"] for s in sessions]))), 3))

## 4. (Tùy chọn) Dữ liệu thật từ ExamGuard
Trang **Thống kê → "Xuất đặc trưng + nhãn (giả danh)"** tạo CSV: mỗi dòng một cửa sổ; `window_label` = 1 (giám thị xác
nhận cờ) / 0 (bác cờ hoặc phiên được rà soát là sạch) / trống (chưa biết). Tải CSV lên Drive rồi đặt đường dẫn.

In [ ]:
PLATFORM_CSV = OUT / "features_exam.csv"
platform_sessions = []
if PLATFORM_CSV.exists():
    pdf = pd.read_csv(PLATFORM_CSV)
    for sid, g in pdf.groupby("session"):
        g = g.sort_values("t")
        vals = g[FEATURES].values.astype(np.float32)
        mk = g[[f"mask_{m}" for m in MODALITIES]].values.astype(np.float32)
        X = np.concatenate([np.clip(vals / SCALES, -5, 5) * np.repeat(mk, [MOD_SLICES[m].stop - MOD_SLICES[m].start for m in MODALITIES], 1), mk], 1)
        yl = g.window_label.values
        platform_sessions.append({"id": f"exam:{sid}", "X": X, "y": np.nan_to_num(yl.astype(float), nan=-1).astype(np.int64),
                                  "tier": g.tier.iloc[0], "source": "platform"})
    print(len(platform_sessions), "phiên thật từ ExamGuard (nhãn -1 = chưa biết)")

## 5. Huấn luyện
- **Tiền huấn luyện tự giám sát:** che ngẫu nhiên một phương thức, học tái tạo đặc trưng bị che (không cần nhãn).
- **Tinh chỉnh MIL:** chia phiên thành đoạn 150 cửa sổ (5 phút). Mất mát = BCE theo cửa sổ (nơi có nhãn) + BCE mức túi
  (túi dương nếu đoạn có gian lận, điểm túi = log-sum-exp mềm của điểm cửa sổ).
- **Modality dropout:** mỗi đoạn tắt ngẫu nhiên từng phương thức với xác suất 0,3 (bắt buộc vì máy yếu / thiếu tín hiệu).

In [ ]:
from torch import nn

CHUNK = 150
BATCH = 32
NF = len(FEATURES)
MOD_WIDTH = [MOD_SLICES[m].stop - MOD_SLICES[m].start for m in MODALITIES]


def chunks(sess_list, chunk=CHUNK, stride=CHUNK // 2):
    xs, ys = [], []
    for s in sess_list:
        X, y = s["X"], s["y"]
        for a in range(0, max(1, len(X) - chunk + 1), stride):
            xc, yc = X[a:a + chunk], y[a:a + chunk]
            if len(xc) < chunk:
                xc = np.concatenate([xc, np.zeros((chunk - len(xc), DIM), np.float32)])
                yc = np.concatenate([yc, -np.ones(chunk - len(yc), np.int64)])
            xs.append(xc)
            ys.append(yc)
    return torch.tensor(np.stack(xs)), torch.tensor(np.stack(ys))


def modality_dropout(x, p=0.3):
    if p <= 0:
        return x
    B = x.shape[0]
    drop = (torch.rand(B, 1, len(MODALITIES), device=x.device) < p).float()
    keep = 1 - drop
    feat_keep = torch.repeat_interleave(keep, torch.tensor(MOD_WIDTH, device=x.device), dim=-1)
    x = x.clone()
    x[..., :NF] *= feat_keep
    x[..., NF:] *= keep
    return x


def pretrain(model, X, epochs=3, lr=2e-3):
    opt = torch.optim.AdamW(model.parameters(), lr)
    for ep in range(epochs):
        tot = 0
        for idx in torch.randperm(len(X)).split(BATCH):
            x = X[idx].to(DEV)
            m = torch.randint(0, len(MODALITIES), (len(x),), device=DEV)
            keep = torch.ones(len(x), 1, len(MODALITIES), device=DEV)
            keep[torch.arange(len(x)), 0, m] = 0
            feat_keep = torch.repeat_interleave(keep, torch.tensor(MOD_WIDTH, device=DEV), -1)
            xin = x.clone()
            xin[..., :NF] *= feat_keep
            xin[..., NF:] *= keep
            _, _, rec = model(xin)
            target_mask = (1 - feat_keep) * torch.repeat_interleave(x[..., NF:], torch.tensor(MOD_WIDTH, device=DEV), -1)
            loss = (((rec - x[..., :NF]) ** 2) * target_mask).sum() / target_mask.sum().clamp(min=1)
            opt.zero_grad()
            loss.backward()
            opt.step()
            tot += loss.item()
        print(f"  tiền huấn luyện {ep + 1}/{epochs}: {tot:.3f}")


def finetune(model, X, Y, epochs=10, lr=2e-3, p_drop=0.3, bag_w=1.0):
    opt = torch.optim.AdamW(model.parameters(), lr, weight_decay=1e-4)
    pos = (Y == 1).float().sum()
    neg = (Y == 0).float().sum()
    pw = torch.tensor(float(neg / pos.clamp(min=1)), device=DEV).clamp(max=20)
    for ep in range(epochs):
        model.train()
        tot = 0
        for idx in torch.randperm(len(X)).split(BATCH):
            x, y = modality_dropout(X[idx].to(DEV), p_drop), Y[idx].to(DEV)
            logits, _, _ = model(x)
            lab = y >= 0
            w_loss = nn.functional.binary_cross_entropy_with_logits(logits[lab], y[lab].float(), pos_weight=pw) if lab.any() else 0
            bag_y = (y == 1).any(1).float()
            bag_known = lab.any(1)
            bag_logit = torch.logsumexp(logits.masked_fill(~lab, -1e4), 1) - math.log(CHUNK)
            b_loss = nn.functional.binary_cross_entropy_with_logits(bag_logit[bag_known], bag_y[bag_known]) if bag_known.any() else 0
            loss = w_loss + bag_w * b_loss
            opt.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            tot += float(loss)
        print(f"  tinh chỉnh {ep + 1}/{epochs}: {tot:.3f}")
    return model.eval()


@torch.no_grad()
def predict_session(model, X: np.ndarray) -> np.ndarray:
    """Streaming từ đầu đến cuối phiên (giống server)."""
    logits, _, _ = model(torch.tensor(X[None]).to(DEV))
    return torch.sigmoid(logits)[0].cpu().numpy()


def server_risk(p: np.ndarray, decay=0.9) -> np.ndarray:
    """Giống backend: risk_t = max(decay·risk_{t-1}, 100·p_t)."""
    r, out = 0.0, []
    for v in p:
        r = max(decay * r, 100 * float(v))
        out.append(r)
    return np.array(out)

## 6. Đánh giá chéo theo người (5 fold) + baseline luật ngưỡng + ablation

In [ ]:
from sklearn.model_selection import GroupKFold

RUN_ABLATIONS = True
groups = np.array([s["id"] for s in sessions])


def run_fold(train_s, test_s, pretrain_on=True, p_drop=0.3, only_modality=None, no_gate=False):
    model = build_fusion_model().to(DEV)
    if no_gate:
        model.gate.weight.data.zero_()
        model.gate.bias.data.zero_()
        model.gate.requires_grad_(False)
    def restrict(sl):
        if only_modality is None:
            return sl
        out = []
        for s in sl:
            X = s["X"].copy()
            for k, m in enumerate(MODALITIES):
                if m != only_modality:
                    X[:, MOD_SLICES[m]] = 0
                    X[:, NF + k] = 0
            out.append({**s, "X": X})
        return out
    train_s, test_s = restrict(train_s), restrict(test_s)
    Xc, Yc = chunks(train_s + platform_sessions)
    if pretrain_on:
        pretrain(model, Xc)
    finetune(model, Xc, Yc, p_drop=p_drop)
    return model, [(s, predict_session(model, s["X"])) for s in test_s]


def collect(preds, key="model"):
    yw, sw, ys, ss, rows = [], [], [], [], []
    for s, p in preds:
        m = s["y"] >= 0
        yw.append(s["y"][m])
        sw.append(p[m])
        ys.append(int((s["y"] == 1).any()))
        ss.append(float(server_risk(p).max()))
    return {"window": summarize(np.concatenate(yw), np.concatenate(sw)), "session": summarize(np.array(ys), np.array(ss))}


configs = {"Hợp nhất (đầy đủ)": {}}
if RUN_ABLATIONS:
    configs.update({"Bỏ tiền huấn luyện": {"pretrain_on": False}, "Bỏ modality dropout": {"p_drop": 0.0},
                    "Bỏ cổng hợp nhất": {"no_gate": True}})
    configs.update({f"Chỉ {m}": {"only_modality": m} for m in ["face", "object", "audio", "browser", "keystroke"]})

all_preds = {k: [] for k in configs}
rule_preds = []
gkf = GroupKFold(n_splits=5)
for fold, (tr, te) in enumerate(gkf.split(sessions, groups=groups)):
    print(f"Fold {fold + 1}/5")
    tr_s = [sessions[i] for i in tr]
    te_s = [sessions[i] for i in te]
    for name, kw in configs.items():
        _, preds = run_fold(tr_s, te_s, **kw)
        all_preds[name] += preds
    rule_preds += [(s, rule_score_matrix(s["X"])) for s in te_s]

rows = []
for name, preds in [("Baseline luật ngưỡng", rule_preds)] + list(all_preds.items()):
    r = collect(preds)
    rows.append({"cấu hình": name, **{f"cửa sổ {k}": v for k, v in r["window"].items() if k not in ("n", "pos")},
                 **{f"phiên {k}": v for k, v in r["session"].items() if k not in ("n", "pos")}})
report = pd.DataFrame(rows)
report.to_csv(OUT / "results_fusion.csv", index=False)
report

### Theo nguồn dữ liệu và theo kiểu gian lận (mô hình đầy đủ)

In [ ]:
by = []
for src in ("oep", "synthetic"):
    sub = [(s, p) for s, p in all_preds["Hợp nhất (đầy đủ)"] if s["source"] == src]
    if sub:
        by.append({"nguồn": src, **collect(sub)["window"]})
for kind in EPISODES:
    sub = [(s, p) for s, p in all_preds["Hợp nhất (đầy đủ)"] if s["source"] == "synthetic" and (kind in s.get("kinds", []) or not s["y"].any())]
    by.append({"kiểu": kind, **collect(sub)["session"]})
pd.DataFrame(by)

## 7. Ngưỡng theo hạng máy (C5)
Ngưỡng chung có thể làm máy yếu bị gắn cờ nhầm nhiều hơn. Tìm ngưỡng điểm rủi ro (thang 0–100 như server) để FPR mức phiên
bằng nhau giữa các hạng máy, rồi đặt vào `backend/.env`: `TIER_THRESHOLDS=low:..,mid:..,high:..`.

In [ ]:
TARGET_FPR = 0.05
preds = all_preds["Hợp nhất (đầy đủ)"]
tiers = {}
for tier in ("low", "mid", "high"):
    neg = [server_risk(p).max() for s, p in preds if s["tier"] == tier and not (s["y"] == 1).any()]
    pos = [server_risk(p).max() for s, p in preds if s["tier"] == tier and (s["y"] == 1).any()]
    if len(neg) < 10:
        continue
    thr = float(np.quantile(neg, 1 - TARGET_FPR))
    global_fpr = float(np.mean(np.array(neg) >= 60))
    tiers[tier] = {"threshold": round(thr, 1), "fpr@60": round(global_fpr, 3), "fpr@thr": round(float(np.mean(np.array(neg) >= thr)), 3),
                   "recall@thr": round(float(np.mean(np.array(pos) >= thr)), 3) if pos else None}
print(pd.DataFrame(tiers).T)
print("\nTIER_THRESHOLDS=" + ",".join(f"{k}:{v['threshold']}" for k, v in tiers.items()))

## 8. Huấn luyện mô hình cuối trên toàn bộ dữ liệu và xuất ONNX cho server
Server (`backend/app/services/risk.py`) nhận: input `x[1, 62]`, `h[1, 64]` → output `prob`, `h_out`, `gates`.
Ô dưới còn kiểm tra ONNX chạy từng bước cho kết quả khớp PyTorch chạy cả chuỗi.

In [ ]:
final = build_fusion_model().to(DEV)
Xc, Yc = chunks(sessions + platform_sessions)
pretrain(final, Xc)
finetune(final, Xc, Yc)
final = final.cpu().eval()
onnx_path = str(OUT / "fusion_gru.onnx")
export_fusion_onnx(final, onnx_path)
err = check_onnx_parity(final, onnx_path, sessions[0]["X"][:200])
print("Sai khác ONNX vs PyTorch:", err)
json.dump({"report": report.to_dict(orient="records"), "tiers": tiers, "n_sessions": len(sessions),
           "oep_sessions": len(oep_sessions), "platform_sessions": len(platform_sessions)},
          open(OUT / "fusion_report.json", "w"), ensure_ascii=False, indent=1, default=str)
print("Chép file này vào ExamGuard/backend/models/fusion_gru.onnx:", onnx_path)